# Supervised Learning: a real example
**Intro to ML & AI** — companion notebook to the *Supervised Learning* slides.

The slides used apples and oranges. Here we use **real medical data**: measurements of 569 breast tumours
from microscope images, each labelled as **benign** (not dangerous) or **malignant** (cancer).

Can a perceptron learn to tell them apart?

1. Look at the data
2. Train a perceptron
3. See the decision boundary
4. Logistic regression: how sure is the model? (optional)
5. Use all the features

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

## 1. Look at the data
We start with just **two features**, so we can plot them:
- `mean radius`: the average size of the cell nuclei
- `mean concave points`: how many dents the outline of the nuclei has

In [ ]:
data = load_breast_cancer()
names = list(data.feature_names)
X = data.data[:, [names.index("mean radius"), names.index("mean concave points")]]
y = data.target                       # 0 = malignant, 1 = benign

print("samples:", X.shape[0])
print("malignant:", (y == 0).sum(), "  benign:", (y == 1).sum())

In [ ]:
plt.scatter(X[y == 1, 0], X[y == 1, 1], c="teal", s=12, label="benign (1)")
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="tomato", s=12, label="malignant (0)")
plt.xlabel("mean radius")
plt.ylabel("mean concave points")
plt.legend()
plt.show()

Malignant tumours tend to be **larger** with **more dents**. The groups overlap a little, so no straight line will be perfect.

## 2. Train a perceptron
Same steps as always: split, scale, fit, score.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

model = make_pipeline(StandardScaler(), Perceptron(random_state=0))
model.fit(X_train, y_train)

print(f"accuracy on training data: {model.score(X_train, y_train):.1%}")
print(f"accuracy on test data:     {model.score(X_test, y_test):.1%}")

In [ ]:
# what did it learn? Just two weights and a bias (on scaled features)
perceptron = model[-1]
print("weights w:", perceptron.coef_[0].round(2))
print("bias b:   ", perceptron.intercept_.round(2))

Both weights are **negative**: a larger radius and more dents push the answer towards 0 (malignant).

## 3. See the decision boundary
This helper colours every spot in the plot by what the model would predict there.

In [ ]:
def plot_boundary(model, X, y, title):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                         np.linspace(X[:, 1].min() - 0.01, X[:, 1].max() + 0.01, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.25, cmap="RdYlGn")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], c="teal", s=12, label="benign")
    plt.scatter(X[y == 0, 0], X[y == 0, 1], c="tomato", s=12, label="malignant")
    plt.xlabel("mean radius")
    plt.ylabel("mean concave points")
    plt.title(title)
    plt.legend()
    plt.show()

plot_boundary(model, X_test, y_test, "Perceptron: test data")

A straight line, just like in the slides. The points on the wrong side are the mistakes.

🧪 **Try it:** Train a `KNeighborsClassifier(n_neighbors=15)` instead (swap it into the pipeline) and plot its boundary.
Is the boundary still straight? Is the test accuracy better or worse?

In [ ]:
# your code here

## 4. Logistic regression: how sure is the model? (optional)
Logistic regression also draws a straight line, but it gives a **probability**.
For a doctor, "92 % sure it is benign" is more useful than just "benign".

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression())
logreg.fit(X_train, y_train)
print(f"accuracy on test data: {logreg.score(X_test, y_test):.1%}")

# three new tumours: small and smooth, medium, large with many dents
new = np.array([[11.0, 0.02],
                [15.0, 0.05],
                [20.0, 0.12]])
for tumour, p in zip(new, logreg.predict_proba(new)):
    print(f"radius {tumour[0]:4.1f}, concave points {tumour[1]:.2f}  ->  {p[1]:.0%} benign, {p[0]:.0%} malignant")

The middle tumour lies near the line, so the model is less sure. That is exactly when a human expert should take a closer look.

## 5. Use all the features
The dataset has **30 features**. We can't plot 30 dimensions, but the model doesn't care: it just learns 30 weights.

In [ ]:
X_all = data.data
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all, y, test_size=0.25, random_state=0, stratify=y)

for name, clf in [("Perceptron", Perceptron(random_state=0)), ("Logistic regression", LogisticRegression())]:
    m = make_pipeline(StandardScaler(), clf).fit(X_train_all, y_train_all)
    print(f"{name:20s} 2 features -> 30 features: test accuracy {m.score(X_test_all, y_test_all):.1%}")

More (useful) features give the model more information, and the accuracy goes up.

🧪 **Try it:** Train the perceptron on all 30 features **with** and **without** `StandardScaler`.
Try a few different starts: `random_state=0, 1, 2, 3, 4`. What do you notice? Why?

In [ ]:
# your code here

## Summary
- A perceptron learns **weights and a bias** from labelled examples, and draws a **straight** decision boundary.
- k-NN can draw a **curved** boundary.
- Logistic regression also gives a **probability**: useful when a person makes the final decision.
- Always **scale** the features and judge the model on **test data**.

⚠️ This is a teaching example. Real medical tools need much more testing, and a doctor always makes the decision.